## 1. Import

In [1]:
import pandas as pd
import numpy as np
import pickle

## 2. Artifacts loading

In [2]:
with open('hotels_preprocessed.pkl', 'rb') as file:
    hotels_preprocessed = pickle.load(file)

df_hotels = pd.DataFrame(hotels_preprocessed)

with open('final_similarity_matrix.pkl', 'rb') as file:
    final_similarity_matrix = pickle.load(file)

In [3]:
with open('feature_similarity_norm.pkl', 'rb') as file:
    feature_similarity_norm = pickle.load(file)

with open('embedding_similarity_norm.pkl', 'rb') as file:
    embedding_similarity_norm = pickle.load(file)

In [4]:
print('Hotels dataframe shape:', df_hotels.shape)
print('Final similarity matrix shape:', final_similarity_matrix.shape)

Hotels dataframe shape: (2470, 72)
Final similarity matrix shape: (2470, 2470)


In [5]:
df_hotels.shape[0] == final_similarity_matrix.shape[0] == final_similarity_matrix.shape[1]

True

In [6]:
id_to_index = {}

for index, hotel_id in enumerate(df_hotels['id']):
    id_to_index[hotel_id] = index

## 3. Recommendation function

### 3.1 Helper functions

In [7]:
# List of columns with service groups
service_cols = [
    'common_services',
    'parking',
    'room_amenities',
    'special_rooms',
    'food_services',
    'transport',
    'laundry_services',
    'entertainment',
    'health_and_beauty',
    'pool',
    'beach_line',
    'sports',
    'business_services',
    'children_services',
    'beach_services',
    'beach_type',
    'hotel_type',
    'internet'
]

In [8]:
def get_hotel_index(hotel_id):
    if hotel_id not in id_to_index:
        print('Такого hotel_id нет в датасете')
        return None
    
    hotel_index = id_to_index[hotel_id]
    
    return hotel_index

In [9]:
def create_candidates_table(hotel_id, similarity_matrix=None):
    if similarity_matrix is None:
        similarity_matrix = final_similarity_matrix
    
    hotel_index = get_hotel_index(hotel_id)
    
    if hotel_index is None:
        return None
    
    similarity_scores = similarity_matrix[hotel_index]
    
    candidates = df_hotels.copy()
    candidates['similarity_score'] = similarity_scores
    
    candidates = candidates[candidates['id'] != hotel_id].copy()
    
    candidates = candidates.sort_values(
        by='similarity_score',
        ascending=False
    ).reset_index(drop=True)
    
    return candidates

In [10]:
def add_candidates_by_filter(candidates, selected_ids, condition, stage_name, needed_count):
    stage_candidates = candidates[condition].copy()
    
    stage_candidates = stage_candidates[
        ~stage_candidates['id'].isin(selected_ids)
    ].copy()
    
    stage_candidates = stage_candidates.head(needed_count).copy()
    
    stage_candidates['selection_stage'] = stage_name
    
    return stage_candidates

In [11]:
service_info_cols = []

for col in service_cols:
    service_info_cols.append(col + '_has_info')


def count_shared_service_info(source_hotel, candidate_hotel):
    shared_count = 0
    
    for col in service_info_cols:
        if source_hotel[col] == 1 and candidate_hotel[col] == 1:
            shared_count += 1
    
    return shared_count


def add_data_coverage_metrics(candidates, source_hotel):
    candidates = candidates.copy()
    
    comparison_data_shares = []
    
    for index, candidate_hotel in candidates.iterrows():
        shared_count = count_shared_service_info(source_hotel, candidate_hotel)
        comparison_share = shared_count / len(service_info_cols)
        comparison_data_shares.append(comparison_share)
    
    candidates['comparison_data_share'] = comparison_data_shares
    
    return candidates

### 3.2 recommend_hotels

In [12]:
def recommend_hotels(hotel_id, top_n=5, similarity_matrix=None):
    if similarity_matrix is None:
        similarity_matrix = final_similarity_matrix
    
    hotel_index = get_hotel_index(hotel_id)
    
    if hotel_index is None:
        return None
    
    source_hotel = df_hotels.loc[hotel_index]
    
    candidates = create_candidates_table(
        hotel_id=hotel_id,
        similarity_matrix=similarity_matrix
    )
    
    selected_candidates = []
    selected_ids = []
    
    same_geo = (
        candidates['geo_cluster'] == source_hotel['geo_cluster']
    )
    
    same_category = (
        candidates['category_clean'] == source_hotel['category_clean']
    )
    
    same_known_stars = (
        (source_hotel['hotel_stars_for_model'] != 'unknown') &
        (candidates['hotel_stars_for_model'] != 'unknown') &
        (candidates['hotel_stars_for_model'] == source_hotel['hotel_stars_for_model'])
    )
    
    stages = [
        ('same_geo_known_stars_category', same_geo & same_known_stars & same_category),
        ('same_geo_known_stars', same_geo & same_known_stars),
        ('same_geo_category', same_geo & same_category),
        ('same_geo', same_geo),
        ('same_known_stars_category', same_known_stars & same_category),
        ('most_similar_overall', candidates['id'].notna())
    ]
    
    for stage_name, condition in stages:
        if len(selected_ids) >= top_n:
            break
        
        needed_count = top_n - len(selected_ids)
        
        stage_candidates = add_candidates_by_filter(
            candidates=candidates,
            selected_ids=selected_ids,
            condition=condition,
            stage_name=stage_name,
            needed_count=needed_count
        )
        
        if len(stage_candidates) > 0:
            selected_candidates.append(stage_candidates)
            selected_ids.extend(stage_candidates['id'].tolist())
    
    if len(selected_candidates) == 0:
        return pd.DataFrame()
    
    result = pd.concat(selected_candidates, ignore_index=True)
    result = result.head(top_n).copy()
    
    result = add_data_coverage_metrics(result, source_hotel)
    
    result['similarity_score'] = result['similarity_score'].round(4)
    result['comparison_data_share'] = result['comparison_data_share'].round(4)
    
    result['stars'] = result['hotel_stars_for_model'].replace({
        'unknown': 'not specified'
    })
    
    output_cols = [
        'id',
        'name',
        'similarity_score',
        'geo_cluster',
        'hotel_stars_for_model',
        'category_clean',
        'selection_stage',
        'rooms_count',
        'comparison_data_share'
    ]
    
    return result[output_cols]

In [13]:
recommend_hotels(847, 5)

,id,name,similarity_score,geo_cluster,hotel_stars_for_model,category_clean,selection_stage,rooms_count,comparison_data_share
0,23362,Rixos Sungate The Land Of Legends Free Access,0.9455,1,5,star_hotel,same_geo_known_stars_category,1079,1.0000
1,968,Rixos Premium Tekirova,0.8775,1,5,star_hotel,same_geo_known_stars_category,627,0.9444
2,891,Regnum Carya,0.8667,1,5,star_hotel,same_geo_known_stars_category,612,1.0000
3,23144,Rixos Park,0.8558,1,5,star_hotel,same_geo_known_stars_category,358,0.9444
4,948,Rixos Downtown Antalya,0.8556,1,5,star_hotel,same_geo_known_stars_category,360,0.9444


## 4. Compare function

### 4.1 Helper functions

In [14]:
def is_no_information(value):
    if isinstance(value, list):
        return len(value) == 0
    
    if pd.isna(value):
        return True
    
    if value == 'unknown':
        return True
    
    if value == 'not specified':
        return True
    
    return False


def value_to_string(value):
    if is_no_information(value):
        return 'not enough information'
    
    if isinstance(value, list):
        return ', '.join(value)
    
    return str(value)


def list_match_score(list_1, list_2):
    if not isinstance(list_1, list):
        list_1 = []
    
    if not isinstance(list_2, list):
        list_2 = []

    if len(list_1) == 0 or len(list_2) == 0:
        return 'not enough information'
    
    set_1 = set(list_1)
    set_2 = set(list_2)
    
    common_values = set_1.intersection(set_2)
    
    match = 2 * len(common_values) / (len(set_1) + len(set_2))
    match = round(match * 100)
    
    return str(match) + '%'


def scalar_match_score(value_1, value_2):
    if is_no_information(value_1) or is_no_information(value_2):
        return 'not enough information'
    
    if value_1 == value_2:
        return '100%'
    
    return '0%'

### 4.2 compare_hotels

In [15]:
def compare_hotels(hotel_id_1, hotel_id_2):
    index_1 = get_hotel_index(hotel_id_1)
    index_2 = get_hotel_index(hotel_id_2)
    
    if index_1 is None or index_2 is None:
        return None
    
    hotel_1 = df_hotels.loc[index_1]
    hotel_2 = df_hotels.loc[index_2]
    
    scalar_cols = [
        'category_clean',
        'hotel_stars_for_model',
        'pets_policy',
        'geo_cluster',
        'rooms_count'
    ]
    
    compare_cols = scalar_cols + service_cols
    
    result_rows = []
    
    for col in compare_cols:
        value_1 = hotel_1[col]
        value_2 = hotel_2[col]
        
        if col == 'rooms_count':
            if hotel_1['rooms_count_missing'] == 1 or hotel_2['rooms_count_missing'] == 1:
                match = 'not enough information'
            else:
                match = scalar_match_score(value_1, value_2)
        
        elif col in service_cols:
            match = list_match_score(value_1, value_2)
        
        else:
            match = scalar_match_score(value_1, value_2)
        
        row = {
            'атрибуты': col,
            str(hotel_id_1): value_to_string(value_1),
            str(hotel_id_2): value_to_string(value_2),
            'match': match
        }
        
        result_rows.append(row)
    
    result = pd.DataFrame(result_rows)
    
    return result

In [16]:
compare_hotels(847, 900)

,атрибуты,847,900,match
0,category_clean,star_hotel,star_hotel,100%
1,hotel_stars_for_model,5,5,100%
2,pets_policy,no_pets,not enough information,not enough information
3,geo_cluster,1,1,100%
4,rooms_count,1079,612,0%
5,common_services,"vip-услуги, банкетный зал, консьерж, круглосут...","vip-услуги, магазин сувениров, магазины в отел...",62%
6,parking,частная,частная,100%
7,room_amenities,"душ, кондиционер, мини-бар, сейф, телевизор, т...","душ, кондиционер, мини-бар, обслуживание в ном...",88%
8,special_rooms,"для некурящих, подходит для маломобильных люде...","подходит для маломобильных людей, семейные",80%
9,food_services,"бар, бар у бассейна, вегетарианское меню, диет...","бар, бар у бассейна, вегетарианское меню, диет...",90%


## 5. Testing

### 5.1. Basic Recommendation Test

In [17]:
test_hotel_id = df_hotels.loc[10, 'id']

recommendations = recommend_hotels(test_hotel_id, top_n=5)

recommendations

,id,name,similarity_score,geo_cluster,hotel_stars_for_model,category_clean,selection_stage,rooms_count,comparison_data_share
0,19649,Asel Didim,0.8819,2,3,star_hotel,same_geo_known_stars_category,86,0.5556
1,19604,Atrium,0.8525,2,3,star_hotel,same_geo_known_stars_category,57,0.4444
2,17773,Salinas Bodrum,0.8445,2,3,star_hotel,same_geo_known_stars_category,49,0.5556
3,12583,Kriss,0.8258,2,3,star_hotel,same_geo_known_stars_category,45,0.6667
4,18748,Gumbet Anil,0.8131,2,3,star_hotel,same_geo_known_stars_category,105,0.5556


In [18]:
df_hotels.loc[[id_to_index[test_hotel_id]], [
    'id',
    'name',
    'category_clean',
    'hotel_stars_for_model',
    'geo_cluster',
    'rooms_count',
    'canonical_text'
]]

,id,name,category_clean,hotel_stars_for_model,geo_cluster,rooms_count,canonical_text
10,23430,Afytos Bodrum City,star_hotel,3,2,82,hotel name: Afytos Bodrum City. category: star...


### 5.2 Compare source hotel with first recommendation

In [19]:
first_recommended_id = recommendations.iloc[0]['id']

compare_hotels(test_hotel_id, first_recommended_id)

,атрибуты,23430,19649,match
0,category_clean,star_hotel,star_hotel,100%
1,hotel_stars_for_model,3,3,100%
2,pets_policy,not enough information,not enough information,not enough information
3,geo_cluster,2,2,100%
4,rooms_count,82,86,0%
5,common_services,not enough information,not enough information,not enough information
6,parking,not enough information,not enough information,not enough information
7,room_amenities,"душ, кондиционер, мини-бар, сейф, телевизор, т...","душ, кондиционер, мини-бар, сейф, телевизор, т...",100%
8,special_rooms,not enough information,not enough information,not enough information
9,food_services,"бар у бассейна, ресторан, шведский стол","бар у бассейна, лобби-бар, ресторан, шведский ...",86%


### 5.3 Compare two approaches separately

In [20]:
def compare_feature_and_embedding_methods(hotel_id, top_n=5):
    feature_recs = recommend_hotels(
        hotel_id=hotel_id,
        top_n=top_n,
        similarity_matrix=feature_similarity_norm
    )
    
    embedding_recs = recommend_hotels(
        hotel_id=hotel_id,
        top_n=top_n,
        similarity_matrix=embedding_similarity_norm
    )
    
    feature_ids = set(feature_recs['id'])
    embedding_ids = set(embedding_recs['id'])
    
    common_ids = feature_ids.intersection(embedding_ids)
    overlap_share = len(common_ids) / top_n
    
    comparison_table = pd.DataFrame({
        'rank': range(1, top_n + 1),
        'feature_id': feature_recs['id'].tolist(),
        'feature_name': feature_recs['name'].tolist(),
        'feature_score': feature_recs['similarity_score'].tolist(),
        'embedding_id': embedding_recs['id'].tolist(),
        'embedding_name': embedding_recs['name'].tolist(),
        'embedding_score': embedding_recs['similarity_score'].tolist()
    })
    
    summary = pd.DataFrame({
        'metric': [
            'common_recommendations',
            'overlap_share'
        ],
        'value': [
            len(common_ids),
            round(overlap_share, 2)
        ]
    })
    
    return summary, comparison_table, feature_recs, embedding_recs

In [21]:
test_hotel_id = df_hotels.loc[0, 'id']

summary, comparison_table, feature_recs, embedding_recs = compare_feature_and_embedding_methods(
    test_hotel_id,
    top_n=10
)

In [22]:
summary

,metric,value
0,common_recommendations,4.0
1,overlap_share,0.4


In [23]:
comparison_table

,rank,feature_id,feature_name,feature_score,embedding_id,embedding_name,embedding_score
0,1,11080,Business Life Boutique,0.8666,13974,Glens Palas,0.8400
1,2,13892,Atik Palace,0.8235,5098,Black Tulip,0.8147
2,3,14256,Mula,0.8077,11080,Business Life Boutique,0.8045
3,4,19582,Galata By Boss,0.7989,5135,Aprilis Gold,0.7913
4,5,14210,Mukarnas Taksim,0.7903,5261,Nova Plaza Boutique & Spa,0.7900
5,6,12776,Kate,0.7800,23145,Galatower,0.7839
6,7,15478,Burdock Istanbul,0.7710,19601,Hamit,0.7831
7,8,13972,Gk Regency Suites,0.7709,14210,Mukarnas Taksim,0.7763
8,9,13974,Glens Palas,0.7692,12776,Kate,0.7606
9,10,23490,Joyway Istanbul Sultanahmet,0.7689,11896,The Haze,0.7595
